# Ensemble learning II: boosting as stagewise optimization

**Lecture 15 · Notebook 01 · CMOR 438 / INDE 577**

Bagging fits members independently and averages them. Boosting instead builds an
additive model **sequentially**, with each new learner addressing the current
model's error. Our central question is:

> **How does fitting residuals become gradient descent in function space, and
> how do shrinkage, tree complexity, and stopping control generalization?**

We derive squared-error gradient boosting, inspect a from-scratch implementation,
then compare AdaBoost and modern histogram gradient boosting on real course data.

## How to use this notebook

Run in order with the **Rice DSM** kernel. Before every fit, state what the next
learner receives as its target. The notebook is an experiment report; reusable
stagewise logic lives in `rice_dsm.ml.ensembles`.

**Route:** additive models → loss gradients → residual fitting → shrinkage and
depth → AdaBoost reweighting → production gradient boosting → early stopping →
monitoring and project contribution.

## Learning objectives

You will be able to:

1. distinguish parallel averaging from sequential additive correction;
2. derive negative gradients for squared-error and logistic losses;
3. trace every stage of a small gradient-boosted regressor;
4. explain AdaBoost's example weights and gradient boosting's pseudo-residuals;
5. diagnose learning-rate, depth, stage-count, and early-stopping tradeoffs; and
6. evaluate boosting with held-out, calibration, and operational evidence.

## Why this matters: from weak learning to optimized additive models

Schapire's 1990 result showed how weak PAC learners could be combined into a
strong learner. Freund and Schapire's AdaBoost made that program practical.
Friedman later framed gradient boosting as numerical optimization over functions.
Modern systems such as XGBoost, LightGBM, CatBoost, and histogram gradient
boosting add regularization, efficient split search, missing-value behavior,
category handling, and systems optimization.

These libraries are powerful, but the mechanism remains teachable: choose a
loss, evaluate how the current prediction should change, fit a restricted learner
to that direction, and take a controlled step.

**Professional practice.** Pin the library and model format, document missing
and categorical semantics, validate early-stopping boundaries, and benchmark
CPU/GPU, memory, latency, calibration, and determinism in the target runtime.

In [ ]:
# Import the data-access and plotting tools shared by this laboratory.
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from rice_dsm.paths import course_database_path

RANDOM_STATE = 438
DATABASE_PATH = course_database_path()
assert DATABASE_PATH.is_file(), (
    "Course database missing. From the repository root run: "
    "uv run --locked python scripts/refresh_course.py"
)
sns.set_theme(style="whitegrid", context="notebook")
print("Database:", DATABASE_PATH)
print("Random seed:", RANDOM_STATE)

## 1. The additive model

Boosting constructs
\[
F_M(x)=F_0(x)+\sum_{m=1}^{M}\nu h_m(x),
\]
where $h_m$ is a shallow tree and $0<\nu\le1$ is the learning rate or
**shrinkage**. For empirical risk
\[
\mathcal R(F)=\frac1n\sum_{i=1}^n L(y_i,F(x_i)),
\]
the desired direction at stage $m$ is the negative derivative with respect to
the current prediction:
\[
r_{im}=-\left.\frac{\partial L(y_i,f)}{\partial f}\right|_{f=F_{m-1}(x_i)}.
\]
Fit $h_m(x_i)\approx r_{im}$, then update $F_m=F_{m-1}+\nu h_m$.

For $L(y,f)=\tfrac12(y-f)^2$, the negative gradient is exactly $y-f$: the
ordinary residual. “Fit the residuals” is therefore not a slogan; it is steepest
descent in prediction space for squared-error loss.

## 2. Worked example: diabetes progression regression

The scikit-learn diabetes benchmark contains standardized baseline measurements
and a quantitative disease-progression response one year later. It is real,
small, and historically common, but not sufficient for a current clinical
deployment claim. We query the repository's versioned SQLite copy so data access
is explicit and reproducible.

In [ ]:
# Query observations in deterministic order and inspect the contract.
database_uri = f"file:{DATABASE_PATH.as_posix()}?mode=ro"
with sqlite3.connect(database_uri, uri=True) as connection:
    diabetes = pd.read_sql_query(
        "SELECT * FROM diabetes_observations ORDER BY observation_id", connection
    )

feature_names = ["age", "sex", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6"]
target_name = "disease_progression"
display(diabetes.head())
display(diabetes[[*feature_names, target_name]].describe().T)

assert diabetes.shape == (442, 12)
assert diabetes[feature_names + [target_name]].isna().sum().sum() == 0
assert diabetes["observation_id"].is_unique

In [ ]:
# Inspect response shape and two plausible associations before modeling.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(diabetes[target_name], kde=True, ax=axes[0])
sns.scatterplot(data=diabetes, x="bmi", y=target_name, ax=axes[1])
sns.scatterplot(data=diabetes, x="s5", y=target_name, ax=axes[2])
axes[0].set_title("Response distribution")
axes[1].set_title("Progression versus BMI")
axes[2].set_title("Progression versus s5")
plt.tight_layout()
plt.show()

assert diabetes[target_name].std() > 0
assert diabetes["bmi"].nunique() > 100

In [ ]:
from sklearn.model_selection import train_test_split

# Freeze an untouched regression test audit before tuning stagewise models.
X = diabetes[feature_names].to_numpy()
y = diabetes[target_name].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)
assert len(X_train) + len(X_test) == len(X)
assert set(map(tuple, X_train)).isdisjoint(set(map(tuple, X_test)))

## 3. Trace the first stage by hand

The constant minimizing squared error is $F_0(x)=\bar y$. Therefore the first
pseudo-response is $r_{i1}=y_i-\bar y$. A depth-one tree partitions these
residuals into two regions and predicts each region's mean residual. The updated
prediction moves only a fraction $\nu$ of that correction.

In [ ]:
from rice_dsm.ml.trees import DecisionTreeRegressor

# Reproduce stage one explicitly so every quantity is inspectable.
initial_prediction = y_train.mean()
stage_zero = np.full(y_train.shape, initial_prediction)
residual_one = y_train - stage_zero
first_tree = DecisionTreeRegressor(max_depth=1, min_samples_leaf=10).fit(
    X_train, residual_one
)
learning_rate = 0.1
stage_one = stage_zero + learning_rate * first_tree.predict(X_train)

print("initial constant:", initial_prediction)
print("residual mean:", residual_one.mean())
print("MSE before:", np.mean((y_train-stage_zero)**2))
print("MSE after one stage:", np.mean((y_train-stage_one)**2))
assert np.isclose(residual_one.mean(), 0.0)
assert np.mean((y_train-stage_one)**2) < np.mean((y_train-stage_zero)**2)

In [ ]:
# Visualize how a weak learner supplies only a structured correction.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].scatter(stage_zero, residual_one, alpha=.6)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set(xlabel="current prediction", ylabel="residual",
            title="Stage 0 leaves structured error")
axes[1].scatter(y_train, stage_one, alpha=.6)
limits = [min(y_train.min(), stage_one.min()), max(y_train.max(), stage_one.max())]
axes[1].plot(limits, limits, "--", color="black")
axes[1].set(xlabel="observed response", ylabel="stage-1 prediction",
            title="One shallow tree makes a small correction")
plt.tight_layout(); plt.show()
assert stage_one.shape == y_train.shape

## 4. Inspectable gradient boosting from the course package

`SquaredErrorGradientBoostingRegressor` stores the initial constant, every
residual tree, training loss after every stage, and staged predictions. This is
valuable for teaching and tests. For large or high-stakes workloads, prefer a
mature implementation with explicit version and behavior contracts.

In [ ]:
# Import the inspectable stagewise implementation and response-scale metrics.
from rice_dsm.ml.ensembles import SquaredErrorGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

course_booster = SquaredErrorGradientBoostingRegressor(
    n_estimators=80, learning_rate=0.05, max_depth=2, min_samples_leaf=8
).fit(X_train, y_train)
test_stages = list(course_booster.staged_predict(X_test))
train_rmse = np.sqrt(course_booster.training_loss_)
test_rmse = [np.sqrt(mean_squared_error(y_test, prediction)) for prediction in test_stages]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(np.arange(1, 81), train_rmse, label="training")
ax.plot(np.arange(1, 81), test_rmse, label="test (audit visualization)")
ax.set(xlabel="boosting stage", ylabel="RMSE", title="Stage count is a complexity parameter")
ax.legend(); plt.show()

assert len(course_booster.estimators_) == 80
assert course_booster.training_loss_[-1] < course_booster.training_loss_[0]
assert np.isfinite(test_rmse).all()

### Learning rate and number of stages interact

A smaller $\nu$ makes each update conservative and usually needs more stages.
It is not free regularization: more stages cost fit time, prediction time, and
artifact size. Tree depth controls the complexity of each correction. Stumps
model main effects; deeper trees can encode higher-order interactions but more
readily chase noise.

In a real selection procedure, trace validation—not final test—loss and choose a
stopping rule inside the training boundary.

In [ ]:
# Compare shrinkage schedules on a training-only validation split.
X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE
)
traces = {}
for rate, stages in [(0.5, 30), (0.1, 80), (0.03, 140)]:
    candidate = SquaredErrorGradientBoostingRegressor(
        n_estimators=stages, learning_rate=rate, max_depth=2, min_samples_leaf=8
    ).fit(X_fit, y_fit)
    traces[f"rate={rate}"] = [
        np.sqrt(mean_squared_error(y_valid, pred))
        for pred in candidate.staged_predict(X_valid)
    ]

fig, ax = plt.subplots(figsize=(8, 4.5))
for label, values in traces.items():
    ax.plot(np.arange(1, len(values)+1), values, label=label)
ax.set(xlabel="stage", ylabel="validation RMSE", title="Shrinkage changes the optimization path")
ax.legend(); plt.show()
assert all(len(values) > 0 for values in traces.values())

## 5. AdaBoost: change the example distribution

For binary labels $y_i\in\{-1,+1\}$ and weak predictions
$h_m(x_i)\in\{-1,+1\}$, weighted error is
\[
\epsilon_m=\sum_i w_i\mathbf 1\{y_i\ne h_m(x_i)\}.
\]
If $\epsilon_m<1/2$, set
$\alpha_m=\tfrac12\log((1-\epsilon_m)/\epsilon_m)$ and update
\[
w_i\leftarrow
\frac{w_i\exp[-\alpha_m y_i h_m(x_i)]}{Z_m}.
\]
Misclassified examples gain relative weight. AdaBoost can focus sharply on hard
cases, including mislabeled cases and outliers. The final score is a weighted
vote, not a plain average.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.tree import DecisionTreeClassifier

# Use the real cancer task to inspect the discrete boosting mechanism.
with sqlite3.connect(database_uri, uri=True) as connection:
    cancer = pd.read_sql_query(
        "SELECT * FROM breast_cancer_observations ORDER BY observation_id", connection
    )
cancer_features = ["mean_radius", "mean_texture", "mean_concavity",
                   "worst_radius", "worst_texture", "worst_concave_points"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    cancer[cancer_features].to_numpy(), cancer["diagnosis_code"].to_numpy(),
    test_size=.25, stratify=cancer["diagnosis_code"], random_state=RANDOM_STATE
)
adaboost = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
    n_estimators=80, learning_rate=.5, random_state=RANDOM_STATE,
).fit(Xc_train, yc_train)
# The learned member weights show how much each stump contributes to the vote.
print("first five member weights:", adaboost.estimator_weights_[:5])
print("test balanced accuracy:", balanced_accuracy_score(yc_test, adaboost.predict(Xc_test)))

assert len(adaboost.estimators_) <= 80
assert np.isfinite(adaboost.estimator_weights_).all()

## 6. Logistic loss and classification gradients

For a logit score $F(x)$, $p(x)=\sigma(F(x))$ and binary cross-entropy
\[
L(y,F)=\log(1+e^F)-yF.
\]
Its derivative is $\partial L/\partial F=p-y$, so the negative gradient is
$y-p$. The next learner fits a correction related to probability residuals.
Production gradient-boosting classifiers also compute leaf values, regularize,
and optimize numerically; they are not simply AdaBoost under another name.

## 7. Production comparison: histogram gradient boosting

Histogram methods bin continuous features, evaluate split summaries efficiently,
and often handle larger tabular problems well. Different libraries make
different choices about categories, missing values, regularization, threading,
GPU kernels, and serialization. Read the exact version's documentation.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.dummy import DummyRegressor

production_models = {
    "mean baseline": DummyRegressor(strategy="mean"),
    "random forest": RandomForestRegressor(
        n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "histogram booster": HistGradientBoostingRegressor(
        learning_rate=.05, max_iter=200, max_leaf_nodes=15,
        l2_regularization=1.0, early_stopping=True, random_state=RANDOM_STATE,
    ),
}
# Evaluate every candidate on the same untouched rows and response-scale metrics.
rows = []
for name, model in production_models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    rows.append({"model": name,
                 "MAE": mean_absolute_error(y_test, prediction),
                 "RMSE": np.sqrt(mean_squared_error(y_test, prediction))})
# A table keeps error units visible instead of reducing comparison to rank.
display(pd.DataFrame(rows).set_index("model").round(2))
assert len(rows) == 3

In [ ]:
# Residual diagnostics reveal structure that one aggregate metric hides.
best_model = production_models["histogram booster"]
prediction = best_model.predict(X_test)
residual = y_test - prediction
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
axes[0].scatter(prediction, residual, alpha=.7)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set(xlabel="prediction", ylabel="residual", title="Residuals versus predictions")
sns.histplot(residual, kde=True, ax=axes[1])
axes[1].set_title("Residual distribution")
plt.tight_layout(); plt.show()

assert np.isfinite(residual).all()
assert residual.shape == y_test.shape

## Common failure modes and debugging

- **Training loss falls while validation loss rises:** reduce stage complexity,
  strengthen regularization, or stop using training-only validation evidence.
- **Early stopping leaks:** never use the final test set as the stopping monitor.
- **A tiny learning rate “fixes” everything:** inspect the compensating stage
  count and resource cost.
- **Boosting chases outliers or mislabeled cases:** inspect high-loss rows and
  consider a robust loss or data correction without deleting inconvenient cases.
- **Library upgrade changes predictions:** pin versions and test serialization,
  category, missing-value, and numerical contracts.
- **Importance becomes causality:** boosting predicts associations; it does not
  identify interventions by itself.

## Guided practice

Using the training/validation split, compare maximum depths 1, 2, and 4 while
holding learning rate and stage budget fixed. Plot validation RMSE versus stage.
Explain interaction capacity, optimization speed, and the onset of overfitting.
Then rerun using absolute-error or another robust loss in a production estimator.

In [ ]:
# PRACTICE: make one falsifiable prediction before changing depth.
practice_depth = 1
practice_booster = SquaredErrorGradientBoostingRegressor(
    n_estimators=30, learning_rate=.1, max_depth=practice_depth,
    min_samples_leaf=8,
).fit(X_fit, y_fit)
practice_rmse = np.sqrt(mean_squared_error(y_valid, practice_booster.predict(X_valid)))
print("validation RMSE:", practice_rmse)
assert np.isfinite(practice_rmse)
assert len(practice_booster.training_loss_) == 30

## Independent exercise: student contribution test plan

Add a boosting analysis to the companion student project:

- durable code: `src/dsm_project/algorithms/student/boosting.py`;
- behavior tests: `tests/test_student_boosting.py`;
- small lawful fixture: `data/fixtures/`;
- full student-supplied data: ignored `data/local/` with source, license, schema,
  and checksum recorded;
- applied notebook: imports the package and tells a scientific or operational
  story from question through limitations.

Tests should verify the initial constant, monotone training-loss decrease for a
hand-computable squared-error example, staged prediction count, reproducibility,
failure before fit, feature-count validation, and comparison with a trusted
implementation.

In [ ]:
# Companion-repository workflow (run in its terminal):
# git switch main
# git pull --ff-only
# git switch -c student/boosting-<short-question>
# uv sync --locked
# uv run ruff check .
# uv run pytest -q
# git add src tests notebooks data/fixtures
# git commit -m "Add boosting analysis for <question>"
# git push -u origin student/boosting-<short-question>
assert True

## Extension

Compare XGBoost, LightGBM, CatBoost, and scikit-learn histogram boosting only if
you can document version, licensing, missing and categorical behavior, thread or
GPU configuration, early-stopping boundary, and serialized-artifact test. The
point is an evidence-based systems comparison, not a leaderboard.

## Retrieval practice

1. Why is a residual the negative gradient for half squared-error loss?
2. How do learning rate and stage count interact?
3. What does depth control in a boosted tree sequence?
4. How does AdaBoost's reweighting differ from gradient boosting?
5. Why must early stopping occur inside the training boundary?

## Takeaway

Boosting is sequential additive optimization. Each learner approximates a useful
correction under the chosen loss, and shrinkage controls the step. Its strength
comes with sensitivity to loss, noisy cases, stage complexity, stopping, and
library semantics. Evaluate the full operated artifact, not only its score.

## Further reading

- Freund and Schapire, “A Decision-Theoretic Generalization of On-Line Learning” (1997).
- Friedman, “Greedy Function Approximation: A Gradient Boosting Machine” (2001).
- scikit-learn [gradient boosting guide](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosting).
- Python [`math`](https://docs.python.org/3/library/math.html) documentation for stable numerical primitives.